# Workday → Salesforce Data Integration Pipeline

**Purpose:**  
Reconcile Workday Worker JSON with Salesforce Contacts using deterministic email matching logic.

**Key Functions:**
- Normalize nested Workday JSON
- Construct email identity sets
- Match against Salesforce contacts
- Classify Create vs Update records
- Validate and export upsert files

> Note: This notebook uses placeholder paths. Replace with secure local paths when running.

In [ ]:
import pandas as pd
import json
import os
import matplotlib.pyplot as plt

## Configuration

In [ ]:
BASE_DIR = "path_to_your_project_directory"

WORKDAY_JSON = os.path.join(BASE_DIR, "workday_workers.json")
SALESFORCE_CONTACTS = os.path.join(BASE_DIR, "salesforce_contacts.csv")
OUTPUT_FILE = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.csv")

## Load & Normalize Workday JSON

In [ ]:
with open(WORKDAY_JSON, "r") as f:
    wd_data = json.load(f)

wd = pd.json_normalize(wd_data)
wd.head()

## Load Salesforce Contacts

In [ ]:
sf = pd.read_csv(SALESFORCE_CONTACTS)
sf.head()

## Build Deterministic Email Identity Sets

In [ ]:
def build_email_set(row):
    emails = set()
    primary = row.get("primaryWorkEmail")
    if pd.notna(primary):
        emails.add(str(primary).strip().lower())
    return emails

wd["email_set"] = wd.apply(build_email_set, axis=1)

## Aggregate Salesforce Email Fields

In [ ]:
sf["email_lower"] = sf["Email"].astype(str).str.lower().str.strip()
sf_email_set = set(sf["email_lower"].dropna())

## Cross-System Matching

In [ ]:
wd["Matched_Email"] = wd["email_set"].apply(
    lambda s: next((e for e in s if e in sf_email_set), None)
)

wd["Action__c"] = wd["Matched_Email"].apply(
    lambda x: "Update" if pd.notna(x) else "Create"
)

## Validation Summary

In [ ]:
print("Total Records:", len(wd))
print("Updates:", (wd["Action__c"] == "Update").sum())
print("Creates:", (wd["Action__c"] == "Create").sum())

## Create vs Update Distribution

In [ ]:
counts = wd["Action__c"].value_counts()

plt.figure()
plt.bar(counts.index, counts.values)
plt.title("Salesforce Sync: Create vs Update")
plt.xlabel("Action")
plt.ylabel("Count")
plt.show()

## Export Upsert File

In [ ]:
wd.to_csv(OUTPUT_FILE, index=False)
print("Export complete:", OUTPUT_FILE)